# LR

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

In [ ]:
DATA_PATH = r"D:\墨大sml作业\FeatureA"

feature_path = DATA_PATH + r"\feature_A.csv"

df = pd.read_csv(feature_path)

print("Data shape:", df.shape)
print(df.head())

Data shape: (20000263, 17)
   userId  movieId  user_avg_rating  user_rating_count  user_rating_std  \
0       1        2         3.742857                175         0.382284   
1       1       29         3.742857                175         0.382284   
2       1       32         3.742857                175         0.382284   
3       1       47         3.742857                175         0.382284   
4       1       50         3.742857                175         0.382284   

   user_like_count  user_like_ratio  user_rating_timespan  user_avg_gap_days  \
0               88         0.502857                   204           1.172414   
1               88         0.502857                   204           1.172414   
2               88         0.502857                   204           1.172414   
3               88         0.502857                   204           1.172414   
4               88         0.502857                   204           1.172414   

   item_avg_rating  item_rating_count  it

In [10]:
drop_cols = ["userId", "movieId", "label"]

X_all = df.drop(columns=drop_cols).values
y_all = df["label"].values

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)

X shape: (20000263, 14)
y shape: (20000263,)


In [11]:
def stratified_split_indices(y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)

    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]

    rng.shuffle(idx_0)
    rng.shuffle(idx_1)

    n_test_0 = int(len(idx_0) * test_size)
    n_test_1 = int(len(idx_1) * test_size)

    test_idx = np.concatenate([idx_0[:n_test_0], idx_1[:n_test_1]])
    train_idx = np.concatenate([idx_0[n_test_0:], idx_1[n_test_1:]])

    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return train_idx, test_idx

In [12]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)

    std[std == 0] = 1.0

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

In [13]:
def compute_auc_from_scratch(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    order = np.argsort(y_prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_prob) + 1)

    pos_ranks = ranks[y_true == 1]
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)

    if n_pos == 0 or n_neg == 0:
        return np.nan

    auc = (np.sum(pos_ranks) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auc

In [14]:
def compute_metrics_from_scratch(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (TP + TN) / len(y_true)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    auc = compute_auc_from_scratch(y_true, y_prob)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    }

In [15]:
def tune_lr_C_from_scratch(X_train, y_train, C_values, inner_repeats=3, val_size=0.2, seed=100):
    C_scores = {}

    for C in C_values:
        fold_scores = []

        for r in range(inner_repeats):
            inner_train_idx, val_idx = stratified_split_indices(
                y_train,
                test_size=val_size,
                seed=seed + r
            )

            X_inner_train = X_train[inner_train_idx]
            y_inner_train = y_train[inner_train_idx]

            X_val = X_train[val_idx]
            y_val = y_train[val_idx]

            X_inner_train_scaled, X_val_scaled = standardize_train_test(
                X_inner_train,
                X_val
            )

            model = LogisticRegression(
                C=C,
                max_iter=1000,
                solver="lbfgs",
                random_state=42
            )

            model.fit(X_inner_train_scaled, y_inner_train)

            y_val_pred = model.predict(X_val_scaled)
            y_val_prob = model.predict_proba(X_val_scaled)[:, 1]

            metrics = compute_metrics_from_scratch(
                y_val,
                y_val_pred,
                y_val_prob
            )

            fold_scores.append(metrics["f1"])

        C_scores[C] = np.mean(fold_scores)

    best_C = max(C_scores, key=C_scores.get)

    return best_C, C_scores

In [16]:
C_values = [0.1, 1.0, 10.0]

outer_results = []

for repeat in range(10):
    print(f"\nOuter repeat {repeat + 1}/10")

    train_idx, test_idx = stratified_split_indices(
        y_all,
        test_size=0.2,
        seed=42 + repeat
    )

    X_train = X_all[train_idx]
    y_train = y_all[train_idx]

    X_test = X_all[test_idx]
    y_test = y_all[test_idx]

    best_C, C_scores = tune_lr_C_from_scratch(
        X_train,
        y_train,
        C_values=C_values,
        inner_repeats=3,
        val_size=0.2,
        seed=1000 + repeat * 10
    )

    print("C scores:", C_scores)
    print("Best C:", best_C)

    X_train_scaled, X_test_scaled = standardize_train_test(
        X_train,
        X_test
    )

    final_model = LogisticRegression(
        C=best_C,
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    )

    final_model.fit(X_train_scaled, y_train)

    y_pred = final_model.predict(X_test_scaled)
    y_prob = final_model.predict_proba(X_test_scaled)[:, 1]

    metrics = compute_metrics_from_scratch(
        y_test,
        y_pred,
        y_prob
    )

    metrics["repeat"] = repeat + 1
    metrics["best_C"] = best_C

    outer_results.append(metrics)


Outer repeat 1/10
C scores: {0.1: np.float64(0.7217380364985754), 1.0: np.float64(0.7217380364887793), 10.0: np.float64(0.7217382429410103)}
Best C: 10.0

Outer repeat 2/10
C scores: {0.1: np.float64(0.7215611537746544), 1.0: np.float64(0.7215606599857459), 10.0: np.float64(0.7215609984641759)}
Best C: 0.1

Outer repeat 3/10
C scores: {0.1: np.float64(0.7218432434613513), 1.0: np.float64(0.7218430541007942), 10.0: np.float64(0.7218429221157016)}
Best C: 0.1

Outer repeat 4/10
C scores: {0.1: np.float64(0.7215457956300905), 1.0: np.float64(0.7215451528693994), 10.0: np.float64(0.7215448889477987)}
Best C: 0.1

Outer repeat 5/10
C scores: {0.1: np.float64(0.7218652170910209), 1.0: np.float64(0.7218653660898641), 10.0: np.float64(0.7218652341268327)}
Best C: 1.0

Outer repeat 6/10
C scores: {0.1: np.float64(0.7214876765200319), 1.0: np.float64(0.7214881130350209), 10.0: np.float64(0.7214881130350209)}
Best C: 1.0

Outer repeat 7/10
C scores: {0.1: np.float64(0.7216665621864249), 1.0: np.

In [17]:
results_df = pd.DataFrame(outer_results)

results_df

,accuracy,precision,recall,f1,auc,TP,TN,FP,FN,repeat,best_C
0,0.719257,0.714989,0.728741,0.721800,0.794601,1456814,1420251,580719,542268,1,10.0
1,0.719261,0.714894,0.728982,0.721869,0.794706,1457295,1419788,581182,541787,2,0.1
2,0.719566,0.715426,0.728734,0.722019,0.794960,1456800,1421502,579468,542282,3,0.1
3,0.719073,0.714847,0.728467,0.721593,0.794482,1456266,1420065,580905,542816,4,0.1
4,0.718982,0.714765,0.728358,0.721498,0.794440,1456048,1419917,581053,543034,5,1.0
5,0.719302,0.715174,0.728453,0.721752,0.794708,1456237,1421007,579963,542845,6,1.0
6,0.719069,0.714918,0.728284,0.721539,0.794473,1455900,1420414,580556,543182,7,0.1
7,0.719363,0.715022,0.729015,0.721951,0.794801,1457360,1420128,580842,541722,8,10.0
8,0.719100,0.714995,0.728206,0.721540,0.794589,1455744,1420695,580275,543338,9,10.0
9,0.719476,0.715238,0.728881,0.721995,0.794574,1457092,1420850,580120,541990,10,0.1


In [18]:
metric_cols = ["accuracy", "precision", "recall", "f1", "auc"]

summary_df = pd.DataFrame({
    "mean": results_df[metric_cols].mean(),
    "std": results_df[metric_cols].std()
})

summary_df

,mean,std
accuracy,0.719245,0.000190
precision,0.715027,0.000200
recall,0.728612,0.000295
f1,0.721756,0.000202
auc,0.794633,0.000163


In [19]:
results_path = DATA_PATH + r"\LR_FeatureA_repeated_results.csv"
summary_path = DATA_PATH + r"\LR_FeatureA_summary.csv"

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, encoding="utf-8-sig")

print("Saved:")
print(results_path)
print(summary_path)

Saved:
D:\墨大sml作业\Feature1\LR_FeatureA_repeated_results.csv
D:\墨大sml作业\Feature1\LR_FeatureA_summary.csv
